In [24]:
import numpy as np
import math
import random
import csv
import pandas as pd
import scipy
import scipy.optimize as opt
from bayes_opt import BayesianOptimization
from bayes_opt.logger import JSONLogger
from bayes_opt.event import Events
from bayes_opt.util import load_logs

In [25]:
def read_behavioral_data(n):
    result_stay_cue = []
    result_safe_risk = []
    action_stay_cue = []
    action_safe_risk = []
    if_can_ask = []
    fname = './behavioral_data/uncertainty_' + str(n+1) + '_2022.csv'
    with open(fname,'r') as f :
        for line in f.readlines():
            if line.split(',')[0]=='0' or line.split(',')[0]=='1':
                if int(line.split(',')[3])==0:
                    action_stay_cue.append(0)
                elif int(line.split(',')[3])==1 or int(line.split(',')[3])==2:
                    action_stay_cue.append(1)
                else :
                    print('ERROR')
                result_stay_cue.append(int(line.split(',')[3]))
                result_safe_risk.append(int(float(line.split(',')[6])))
                action_safe_risk.append(int(line.split(',')[4]))
                if line.split(',')[1]==' ':
                    if_can_ask.append(0)
                else :
                    if_can_ask.append(1)

    return if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk

In [26]:
def dir(a):
    a_0 = np.sum(a,axis=0)
    A = np.zeros([8,8])
    for i in range(np.shape(A)[0]):
        for j in range(np.shape(A)[1]):
            A[i,j] = a[i,j]/a_0[j]
    return A
def cum(a):
    a_0 = np.sum(a,axis=0)
    a_cum = np.array([np.ones([1,8])*a_0[0],
                      np.ones([1,8])*a_0[1],
                      np.ones([1,8])*a_0[2],
                      np.ones([1,8])*a_0[3],
                      np.ones([1,8])*a_0[4],
                      np.ones([1,8])*a_0[5],
                      np.ones([1,8])*a_0[6],
                      np.ones([1,8])*a_0[7]]).squeeze().T
    return a_cum
def H_entropy(A):
    H = np.matmul(A.T,np.log(A+np.e**(-16)))
    H = np.diag(H)
    return H
def G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex):
    a_nonzero = np.array([[1,1,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,1,1,0,0,0,0],
                      [0,0,0,0,1,1,0,0],
                      [0,0,0,0,0,0,1,0],
                      [0,0,0,0,0,0,0,1]])
    s = np.array(s)
    discount = 0.1
    o = np.dot(A,s.reshape(-1,1)).reshape(-1)
    w = 1./(cum(a))-1./(a+np.e**(-16))
    w = np.multiply(w,a_nonzero)
    H = -np.dot(H_entropy(A),s.reshape(-1,1))
    AL = np.dot(o,np.dot(w,s.reshape(-1,1)))
    AI = H + np.dot(o,np.log(o+np.e**(-16)).reshape(-1,1))
    preference = np.array([1,2,1.5,0.5,0,0,-0.167,-0.167])
    EX = np.dot(o,preference.reshape(-1,1))
    if action_stay_cue == 0:
        return float(discount*(p_al*AL + p_ai*AI) - p_ex*EX)
    elif action_stay_cue == 1:
        return float(p_al*AL + p_ai*AI - p_ex*EX)
    else :
        print('ERROR')
        
def P_stay_cue(A,a,action_stay_cue,if_can_ask,p_al,p_ai,p_ex):#stay,cue,0,stay-safe,1,stay-risk,2,cue-safe,3,cue-risk
    if if_can_ask == 0:
        return 1
    else:
        s1 = np.array([0,0,0,0,0.5,0.5,0,0])
        s2 = np.array([0.5,0.5,0,0,0,0,0,0])
        G_stay_safe = G_ExperctedFreeEnergy(A,a,s1,0,p_al,p_ai,p_ex)+G_ExperctedFreeEnergy(A,a,s2,0,p_al,p_ai,p_ex)
        s1 = np.array([0,0,0,0,0.5,0.5,0,0])
        s2 = np.array([0,0,0.5,0.5,0,0,0,0])
        G_stay_risk = G_ExperctedFreeEnergy(A,a,s1,0,p_al,p_ai,p_ex)+G_ExperctedFreeEnergy(A,a,s2,0,p_al,p_ai,p_ex)
        s1 = [0, 0, 0, 0, 0, 0, 0.5, 0.5]
        s2 = [1, 0, 0, 0, 0, 0, 0, 0]
        s3 = [0, 0, 0, 0, 0, 0, 0.5, 0.5]
        s4 = [0, 0, 1, 0, 0, 0, 0, 0]
        G_cue_HR_0 = G_ExperctedFreeEnergy(A, a, s1, 1, p_al, p_ai, p_ex) + G_ExperctedFreeEnergy(A, a, s2, 1, p_al, p_ai, p_ex)
        G_cue_HR_1 = G_ExperctedFreeEnergy(A, a, s3, 1, p_al, p_ai, p_ex) + G_ExperctedFreeEnergy(A, a, s4, 1, p_al, p_ai, p_ex)
        if G_cue_HR_0>G_cue_HR_1:
            G_cue_HR = G_cue_HR_1
        else:
            G_cue_HR = G_cue_HR_0
        s1 = [0, 0, 0, 0, 0, 0, 0.5, 0.5]
        s2 = [0, 1, 0, 0, 0, 0, 0, 0]
        s3 = [0, 0, 0, 0, 0, 0, 0.5, 0.5]
        s4 = [0, 0, 0, 1, 0, 0, 0, 0]
        G_cue_LR_0 = G_ExperctedFreeEnergy(A, a, s1, 1, p_al, p_ai, p_ex) + G_ExperctedFreeEnergy(A, a, s2, 1, p_al, p_ai, p_ex)
        G_cue_LR_1 = G_ExperctedFreeEnergy(A, a, s3, 1, p_al, p_ai, p_ex) + G_ExperctedFreeEnergy(A, a, s4, 1, p_al, p_ai, p_ex)
        if G_cue_LR_0>G_cue_LR_1:
            G_cue_LR = G_cue_LR_1
        else:
            G_cue_LR = G_cue_LR_0
        G_stay_cue = np.array([-min(G_stay_safe,G_stay_risk),-(G_cue_HR+G_cue_LR)/2])
        if G_stay_cue[0]>20:
            G_stay_cue[0]=20
        if G_stay_cue[1]>20:
            G_stay_cue[1]=20
        exp_G = [np.exp(G_stay_cue[0]),np.exp(G_stay_cue[1])]
        total = exp_G[0]+exp_G[1]
        P = [exp_G[0]/total,exp_G[1]/total]
        return P[action_stay_cue].squeeze()

def P_safe_risk(A,a,action_safe_risk,result_stay_cue,p_al,p_ai,p_ex):#con=0,no,con=1,HRC,con=2,LRC,pi=0,safe,pi=1,risky
    if result_stay_cue !=0:
        action_stay_cue = 1
    else :
        action_stay_cue = 0
    if result_stay_cue == 0:
        s=np.array([0.5,0.5,0,0,0,0,0,0])
        G_safe = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        G_risk = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
    elif result_stay_cue == 1:
        s=np.array([1,0,0,0,0,0,0,0])
        G_safe = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
        s=np.array([0,0,1,0,0,0,0,0])
        G_risk = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
    else :
        s=np.array([0,1,0,0,0,0,0,0])
        G_safe = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
        s=np.array([0,0,0,1,0,0,0,0])
        G_risk = -G_ExperctedFreeEnergy(A,a,s,action_stay_cue,p_al,p_ai,p_ex)
    if G_safe > 20:
        G_safe = 20
    if G_risk > 20:
        G_risk = 20
    exp_G = [np.exp(G_safe),np.exp(G_risk)]
    total = exp_G[0]+exp_G[1]
    P_safe_risk = [exp_G[0]/total,exp_G[1]/total]
    return P_safe_risk[action_safe_risk].squeeze()

def a_update(a,result_stay_cue,result_safe_risk,action_safe_risk,rate):
    if result_stay_cue==0 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([0.5,0.5,0,0,0,0,0,0])
#safeHRC,safeLRC,riskyHRC,riskLRC,stayHRC,stayLRC,cueHRC,cueLRC
        o=np.array([1,0,0,0,0,0,0,0])
#safe,riskyHR,riskyLR,stay,cueHR,cueLR
    elif result_stay_cue==0 and result_safe_risk==0:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==3:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==9:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==12:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([1,0,0,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==0:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==3:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==9:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])      
    elif result_stay_cue==1 and result_safe_risk==12:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([0,1,0,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==0:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==3:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==9:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])    
    else :
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    if result_stay_cue!=0:
        a=a+rate*np.outer(o,s)#rate:learning rate
    else :
        a=a+rate*0.1*np.outer(o,s)
    return a

def P_active_inference(x):
    trial_num = 120
    subject_num = 25
    if_can_ask_sub = []
    action_stay_cue_sub = []
    result_stay_cue_sub = []
    action_safe_risk_sub = []
    result_safe_risk_sub = []
    for i in range(subject_num):
        if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(i)
        if_can_ask_sub.append(if_can_ask)
        action_stay_cue_sub.append(action_stay_cue)
        result_stay_cue_sub.append(result_stay_cue)
        action_safe_risk_sub.append(action_safe_risk)
        result_safe_risk_sub.append(result_safe_risk)
    rate = []
    prior = []
    p_al = []
    p_ai = []
    p_ex = []
    for i in range(subject_num):
        rate.append(x[i*5])
        prior.append(x[i*5+1])
        p_al.append(x[i*5+2])
        p_ai.append(x[i*5+3])
        p_ex.append(x[i*5+4])
    neg_log_p_policy = 0
    for i in range(subject_num):
        a = np.array([[100.0,100,prior[i],prior[i],0,0,0,0],
                  [0,0,prior[i],prior[i],0,0,0,0],
                  [0,0,prior[i],prior[i],0,0,0,0],
                  [0,0,prior[i],prior[i],0,0,0,0],
                  [0,0,prior[i],prior[i],0,0,0,0],
                  [0,0,0,0,100,100,0,0],
                  [0,0,0,0,0,0,100,0],
                  [0,0,0,0,0,0,0,100]])
        A = dir(a)
        for j in range(trial_num):
            neg_log_p_policy -= np.log(P_stay_cue(A,a,action_stay_cue_sub[i][j],if_can_ask_sub[i][j],p_al[i],p_ai[i],p_ex[i]))
            neg_log_p_policy -= np.log(P_safe_risk(A,a,action_safe_risk_sub[i][j],result_stay_cue_sub[i][j],p_al[i],p_ai[i],p_ex[i]))
            a = a_update(a,result_stay_cue_sub[i][j],result_safe_risk_sub[i][j],action_safe_risk_sub[i][j],rate[i])
            A = dir(a)
    return neg_log_p_policy

In [10]:
def P_active_inference_subject(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 0

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy

In [14]:
def P_active_inference_subject(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 0

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer = BayesianOptimization(
    f=P_active_inference_subject,
    pbounds=pbounds,
    random_state=1,)

In [28]:
def P_active_inference_subject_1(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 0

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_2(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 1

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_3(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 2

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_4(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 3

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_5(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 4

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_6(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 5

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_7(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 6

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_8(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 7

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_9(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 8

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_10(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 9

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_11(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 10

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_12(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 11

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_13(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 12

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_14(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 13

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_15(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 14

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_16(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 15

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_17(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 16

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_18(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 17

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_19(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 18

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_20(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 19

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_21(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 20

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_22(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 21
    
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_23(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 22
    
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_24(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 23
    
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
def P_active_inference_subject_25(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 24
    
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy

In [20]:
def P_active_inference_subject_1(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 0

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_1 = BayesianOptimization(
    f=P_active_inference_subject_1,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_1 = JSONLogger(path="./logs_1.log")
optimizer_1.subscribe(Events.OPTIMIZATION_STEP, logger_1)
optimizer_1.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_active_inference_subject_2(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 1

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_2 = BayesianOptimization(
    f=P_active_inference_subject_2,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_2 = JSONLogger(path="./logs_2.log")
optimizer_2.subscribe(Events.OPTIMIZATION_STEP, logger_2)
optimizer_2.maximize(
    init_points=1000,
    n_iter=1000,
)

In [4]:
def P_active_inference_subject_3(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 2

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_3 = BayesianOptimization(
    f=P_active_inference_subject_3,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_3 = JSONLogger(path="./logs_3.log")
optimizer_3.subscribe(Events.OPTIMIZATION_STEP, logger_3)
optimizer_3.maximize(
    init_points=1000,
    n_iter=1000,
)

In [5]:
def P_active_inference_subject_4(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 3

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_4 = BayesianOptimization(
    f=P_active_inference_subject_4,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_4 = JSONLogger(path="./logs_4.log")
optimizer_4.subscribe(Events.OPTIMIZATION_STEP, logger_4)
optimizer_4.maximize(
    init_points=1000,
    n_iter=1000,
)

In [6]:
def P_active_inference_subject_5(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 4

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_5 = BayesianOptimization(
    f=P_active_inference_subject_5,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_5 = JSONLogger(path="./logs_5.log")
optimizer_5.subscribe(Events.OPTIMIZATION_STEP, logger_5)
optimizer_5.maximize(
    init_points=1000,
    n_iter=1000,
)

In [7]:
def P_active_inference_subject_6(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 5

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_6 = BayesianOptimization(
    f=P_active_inference_subject_6,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_6 = JSONLogger(path="./logs_6.log")
optimizer_6.subscribe(Events.OPTIMIZATION_STEP, logger_6)
optimizer_6.maximize(
    init_points=1000,
    n_iter=1000,
)

In [8]:
def P_active_inference_subject_7(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 6

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_7 = BayesianOptimization(
    f=P_active_inference_subject_7,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_7 = JSONLogger(path="./logs_7.log")
optimizer_7.subscribe(Events.OPTIMIZATION_STEP, logger_7)
optimizer_7.maximize(
    init_points=1000,
    n_iter=1000,
)

In [9]:
def P_active_inference_subject_8(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 7

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_8 = BayesianOptimization(
    f=P_active_inference_subject_8,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_8 = JSONLogger(path="./logs_8.log")
optimizer_8.subscribe(Events.OPTIMIZATION_STEP, logger_8)
optimizer_8.maximize(
    init_points=1000,
    n_iter=1000,
)

In [10]:
def P_active_inference_subject_9(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 8

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_9 = BayesianOptimization(
    f=P_active_inference_subject_9,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_9 = JSONLogger(path="./logs_9.log")
optimizer_9.subscribe(Events.OPTIMIZATION_STEP, logger_9)
optimizer_9.maximize(
    init_points=1000,
    n_iter=1000,
)

In [11]:
def P_active_inference_subject_10(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 9

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_10 = BayesianOptimization(
    f=P_active_inference_subject_10,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_10 = JSONLogger(path="./logs_10.log")
optimizer_10.subscribe(Events.OPTIMIZATION_STEP, logger_10)
optimizer_10.maximize(
    init_points=1000,
    n_iter=1000,
)

In [12]:
def P_active_inference_subject_11(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 10

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_11 = BayesianOptimization(
    f=P_active_inference_subject_11,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_11 = JSONLogger(path="./logs_11.log")
optimizer_11.subscribe(Events.OPTIMIZATION_STEP, logger_11)
optimizer_11.maximize(
    init_points=1000,
    n_iter=1000,
)

In [13]:
def P_active_inference_subject_12(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 11

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_12 = BayesianOptimization(
    f=P_active_inference_subject_12,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_12 = JSONLogger(path="./logs_12.log")
optimizer_12.subscribe(Events.OPTIMIZATION_STEP, logger_12)
optimizer_12.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_active_inference_subject_13(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 12

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_13 = BayesianOptimization(
    f=P_active_inference_subject_13,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_13 = JSONLogger(path="./logs_13.log")
optimizer_13.subscribe(Events.OPTIMIZATION_STEP, logger_13)
optimizer_13.maximize(
    init_points=1000,
    n_iter=1000,
)

In [15]:
def P_active_inference_subject_14(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 13

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_14 = BayesianOptimization(
    f=P_active_inference_subject_14,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_14 = JSONLogger(path="./logs_14.log")
optimizer_14.subscribe(Events.OPTIMIZATION_STEP, logger_14)
optimizer_14.maximize(
    init_points=1000,
    n_iter=1000,
)

In [16]:
def P_active_inference_subject_15(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 14

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_15 = BayesianOptimization(
    f=P_active_inference_subject_15,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_15 = JSONLogger(path="./logs_15.log")
optimizer_15.subscribe(Events.OPTIMIZATION_STEP, logger_15)
optimizer_15.maximize(
    init_points=1000,
    n_iter=1000,
)

In [17]:
def P_active_inference_subject_16(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 15

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_16 = BayesianOptimization(
    f=P_active_inference_subject_16,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_16 = JSONLogger(path="./logs_16.log")
optimizer_16.subscribe(Events.OPTIMIZATION_STEP, logger_16)
optimizer_16.maximize(
    init_points=1000,
    n_iter=1000,
)

In [18]:
def P_active_inference_subject_17(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 16

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_17 = BayesianOptimization(
    f=P_active_inference_subject_17,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_17 = JSONLogger(path="./logs_17.log")
optimizer_17.subscribe(Events.OPTIMIZATION_STEP, logger_17)
optimizer_17.maximize(
    init_points=1000,
    n_iter=1000,
)

Data point [1.e-03 1.e+01 1.e+01 1.e+01 1.e-03] is not unique. 1 duplicates registered. Continuing ...


In [19]:
def P_active_inference_subject_18(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 17

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_18 = BayesianOptimization(
    f=P_active_inference_subject_18,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_18 = JSONLogger(path="./logs_18.log")
optimizer_18.subscribe(Events.OPTIMIZATION_STEP, logger_18)
optimizer_18.maximize(
    init_points=1000,
    n_iter=1000,
)

In [20]:
def P_active_inference_subject_19(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 18

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_19 = BayesianOptimization(
    f=P_active_inference_subject_19,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_19 = JSONLogger(path="./logs_19.log")
optimizer_19.subscribe(Events.OPTIMIZATION_STEP, logger_19)
optimizer_19.maximize(
    init_points=1000,
    n_iter=1000,
)

KeyboardInterrupt: 

In [ ]:
def P_active_inference_subject_20(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 19

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_20 = BayesianOptimization(
    f=P_active_inference_subject_20,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_20 = JSONLogger(path="./logs_20.log")
optimizer_20.subscribe(Events.OPTIMIZATION_STEP, logger_20)
optimizer_20.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_active_inference_subject_21(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 20

    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_21 = BayesianOptimization(
    f=P_active_inference_subject_21,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_21 = JSONLogger(path="./logs_21.log")
optimizer_21.subscribe(Events.OPTIMIZATION_STEP, logger_21)
optimizer_21.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_active_inference_subject_22(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 21
    
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_22 = BayesianOptimization(
    f=P_active_inference_subject_22,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_22 = JSONLogger(path="./logs_22.log")
optimizer_22.subscribe(Events.OPTIMIZATION_STEP, logger_22)
optimizer_22.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_active_inference_subject_23(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 22
    
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_23 = BayesianOptimization(
    f=P_active_inference_subject_23,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_23 = JSONLogger(path="./logs_23.log")
optimizer_23.subscribe(Events.OPTIMIZATION_STEP, logger_23)
optimizer_23.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_active_inference_subject_24(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 23
    
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_24 = BayesianOptimization(
    f=P_active_inference_subject_24,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_24 = JSONLogger(path="./logs_24.log")
optimizer_24.subscribe(Events.OPTIMIZATION_STEP, logger_24)
optimizer_24.maximize(
    init_points=1000,
    n_iter=1000,
)

KeyboardInterrupt: 

In [ ]:
def P_active_inference_subject_25(rate,prior,p_al,p_ai,p_ex):
    trial_num = 120
    subject = 24
    
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p_policy = 0

    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for j in range(trial_num):
        log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue[j],if_can_ask[j],p_al,p_ai,p_ex))
        log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk[j],result_stay_cue[j],p_al,p_ai,p_ex))
        a = a_update(a,result_stay_cue[j],result_safe_risk[j],action_safe_risk[j],rate)
        A = dir(a)
    return log_p_policy
pbounds = {'rate':(0.001,10),'prior':(0.001,10),'p_al':(0.001,10),'p_ai':(0.001,10),'p_ex':(0.001,10)}
optimizer_25 = BayesianOptimization(
    f=P_active_inference_subject_25,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
logger_25 = JSONLogger(path="./logs_25.log")
optimizer_25.subscribe(Events.OPTIMIZATION_STEP, logger_25)
optimizer_25.maximize(
    init_points=1000,
    n_iter=1000,
)

In [47]:
P_AL,P_AI,P_EX,PRIOR,RATE,LL = [],[],[],[],[],[]

optimizer_1 = BayesianOptimization(
    f=P_active_inference_subject_1,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_1, logs=["./logs_1.log.json"])
P_AL.append(optimizer_1.max['params']['p_al'])
P_AI.append(optimizer_1.max['params']['p_ai'])
P_EX.append(optimizer_1.max['params']['p_ex'])
RATE.append(optimizer_1.max['params']['rate'])
PRIOR.append(optimizer_1.max['params']['prior'])
LL.append(optimizer_1.max['target'])

optimizer_2 = BayesianOptimization(
    f=P_active_inference_subject_2,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_2, logs=["./logs_2.log.json"])
P_AL.append(optimizer_2.max['params']['p_al'])
P_AI.append(optimizer_2.max['params']['p_ai'])
P_EX.append(optimizer_2.max['params']['p_ex'])
RATE.append(optimizer_2.max['params']['rate'])
PRIOR.append(optimizer_2.max['params']['prior'])
LL.append(optimizer_2.max['target'])

optimizer_3 = BayesianOptimization(
    f=P_active_inference_subject_3,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_3, logs=["./logs_3.log.json"])
P_AL.append(optimizer_3.max['params']['p_al'])
P_AI.append(optimizer_3.max['params']['p_ai'])
P_EX.append(optimizer_3.max['params']['p_ex'])
RATE.append(optimizer_3.max['params']['rate'])
PRIOR.append(optimizer_3.max['params']['prior'])
LL.append(optimizer_3.max['target'])

optimizer_4 = BayesianOptimization(
    f=P_active_inference_subject_4,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_4, logs=["./logs_4.log.json"])
P_AL.append(optimizer_4.max['params']['p_al'])
P_AI.append(optimizer_4.max['params']['p_ai'])
P_EX.append(optimizer_4.max['params']['p_ex'])
RATE.append(optimizer_4.max['params']['rate'])
PRIOR.append(optimizer_4.max['params']['prior'])
LL.append(optimizer_4.max['target'])

optimizer_5 = BayesianOptimization(
    f=P_active_inference_subject_5,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_5, logs=["./logs_5.log.json"])
P_AL.append(optimizer_5.max['params']['p_al'])
P_AI.append(optimizer_5.max['params']['p_ai'])
P_EX.append(optimizer_5.max['params']['p_ex'])
RATE.append(optimizer_5.max['params']['rate'])
PRIOR.append(optimizer_5.max['params']['prior'])
LL.append(optimizer_5.max['target'])

optimizer_6 = BayesianOptimization(
    f=P_active_inference_subject_6,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_6, logs=["./logs_6.log.json"])
P_AL.append(optimizer_6.max['params']['p_al'])
P_AI.append(optimizer_6.max['params']['p_ai'])
P_EX.append(optimizer_6.max['params']['p_ex'])
RATE.append(optimizer_6.max['params']['rate'])
PRIOR.append(optimizer_6.max['params']['prior'])
LL.append(optimizer_6.max['target'])

optimizer_7 = BayesianOptimization(
    f=P_active_inference_subject_7,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_7, logs=["./logs_7.log.json"])
P_AL.append(optimizer_7.max['params']['p_al'])
P_AI.append(optimizer_7.max['params']['p_ai'])
P_EX.append(optimizer_7.max['params']['p_ex'])
RATE.append(optimizer_7.max['params']['rate'])
PRIOR.append(optimizer_7.max['params']['prior'])
LL.append(optimizer_7.max['target'])

optimizer_8 = BayesianOptimization(
    f=P_active_inference_subject_8,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_8, logs=["./logs_8.log.json"])
P_AL.append(optimizer_8.max['params']['p_al'])
P_AI.append(optimizer_8.max['params']['p_ai'])
P_EX.append(optimizer_8.max['params']['p_ex'])
RATE.append(optimizer_8.max['params']['rate'])
PRIOR.append(optimizer_8.max['params']['prior'])
LL.append(optimizer_8.max['target'])

optimizer_9 = BayesianOptimization(
    f=P_active_inference_subject_9,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_9, logs=["./logs_9.log.json"])
P_AL.append(optimizer_9.max['params']['p_al'])
P_AI.append(optimizer_9.max['params']['p_ai'])
P_EX.append(optimizer_9.max['params']['p_ex'])
RATE.append(optimizer_9.max['params']['rate'])
PRIOR.append(optimizer_9.max['params']['prior'])
LL.append(optimizer_9.max['target'])

optimizer_10 = BayesianOptimization(
    f=P_active_inference_subject_10,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_10, logs=["./logs_10.log.json"])
P_AL.append(optimizer_10.max['params']['p_al'])
P_AI.append(optimizer_10.max['params']['p_ai'])
P_EX.append(optimizer_10.max['params']['p_ex'])
RATE.append(optimizer_10.max['params']['rate'])
PRIOR.append(optimizer_10.max['params']['prior'])
LL.append(optimizer_10.max['target'])

optimizer_11 = BayesianOptimization(
    f=P_active_inference_subject_11,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_11, logs=["./logs_11.log.json"])
P_AL.append(optimizer_11.max['params']['p_al'])
P_AI.append(optimizer_11.max['params']['p_ai'])
P_EX.append(optimizer_11.max['params']['p_ex'])
RATE.append(optimizer_11.max['params']['rate'])
PRIOR.append(optimizer_11.max['params']['prior'])
LL.append(optimizer_11.max['target'])

optimizer_12 = BayesianOptimization(
    f=P_active_inference_subject_12,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_12, logs=["./logs_12.log.json"])
P_AL.append(optimizer_12.max['params']['p_al'])
P_AI.append(optimizer_12.max['params']['p_ai'])
P_EX.append(optimizer_12.max['params']['p_ex'])
RATE.append(optimizer_12.max['params']['rate'])
PRIOR.append(optimizer_12.max['params']['prior'])
LL.append(optimizer_12.max['target'])

optimizer_13 = BayesianOptimization(
    f=P_active_inference_subject_13,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_13, logs=["./logs_13.log.json"])
P_AL.append(optimizer_13.max['params']['p_al'])
P_AI.append(optimizer_13.max['params']['p_ai'])
P_EX.append(optimizer_13.max['params']['p_ex'])
RATE.append(optimizer_13.max['params']['rate'])
PRIOR.append(optimizer_13.max['params']['prior'])
LL.append(optimizer_13.max['target'])

optimizer_14 = BayesianOptimization(
    f=P_active_inference_subject_14,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_14, logs=["./logs_14.log.json"])
P_AL.append(optimizer_14.max['params']['p_al'])
P_AI.append(optimizer_14.max['params']['p_ai'])
P_EX.append(optimizer_14.max['params']['p_ex'])
RATE.append(optimizer_14.max['params']['rate'])
PRIOR.append(optimizer_14.max['params']['prior'])
LL.append(optimizer_14.max['target'])

optimizer_15 = BayesianOptimization(
    f=P_active_inference_subject_15,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_15, logs=["./logs_15.log.json"])
P_AL.append(optimizer_15.max['params']['p_al'])
P_AI.append(optimizer_15.max['params']['p_ai'])
P_EX.append(optimizer_15.max['params']['p_ex'])
RATE.append(optimizer_15.max['params']['rate'])
PRIOR.append(optimizer_15.max['params']['prior'])
LL.append(optimizer_15.max['target'])

optimizer_16 = BayesianOptimization(
    f=P_active_inference_subject_16,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_16, logs=["./logs_16.log.json"])
P_AL.append(optimizer_16.max['params']['p_al'])
P_AI.append(optimizer_16.max['params']['p_ai'])
P_EX.append(optimizer_16.max['params']['p_ex'])
RATE.append(optimizer_16.max['params']['rate'])
PRIOR.append(optimizer_16.max['params']['prior'])
LL.append(optimizer_16.max['target'])

optimizer_17 = BayesianOptimization(
    f=P_active_inference_subject_17,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_17, logs=["./logs_17.log.json"])
P_AL.append(optimizer_17.max['params']['p_al'])
P_AI.append(optimizer_17.max['params']['p_ai'])
P_EX.append(optimizer_17.max['params']['p_ex'])
RATE.append(optimizer_17.max['params']['rate'])
PRIOR.append(optimizer_17.max['params']['prior'])
LL.append(optimizer_17.max['target'])

optimizer_18 = BayesianOptimization(
    f=P_active_inference_subject_18,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_18, logs=["./logs_18.log.json"])
P_AL.append(optimizer_18.max['params']['p_al'])
P_AI.append(optimizer_18.max['params']['p_ai'])
P_EX.append(optimizer_18.max['params']['p_ex'])
RATE.append(optimizer_18.max['params']['rate'])
PRIOR.append(optimizer_18.max['params']['prior'])
LL.append(optimizer_18.max['target'])

optimizer_19 = BayesianOptimization(
    f=P_active_inference_subject_19,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_19, logs=["./logs_19.log.json"])
P_AL.append(optimizer_19.max['params']['p_al'])
P_AI.append(optimizer_19.max['params']['p_ai'])
P_EX.append(optimizer_19.max['params']['p_ex'])
RATE.append(optimizer_19.max['params']['rate'])
PRIOR.append(optimizer_19.max['params']['prior'])
LL.append(optimizer_19.max['target'])

optimizer_20 = BayesianOptimization(
    f=P_active_inference_subject_20,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_20, logs=["./logs_20.log.json"])
P_AL.append(optimizer_20.max['params']['p_al'])
P_AI.append(optimizer_20.max['params']['p_ai'])
P_EX.append(optimizer_20.max['params']['p_ex'])
RATE.append(optimizer_20.max['params']['rate'])
PRIOR.append(optimizer_20.max['params']['prior'])
LL.append(optimizer_20.max['target'])

optimizer_21 = BayesianOptimization(
    f=P_active_inference_subject_21,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_21, logs=["./logs_21.log.json"])
P_AL.append(optimizer_21.max['params']['p_al'])
P_AI.append(optimizer_21.max['params']['p_ai'])
P_EX.append(optimizer_21.max['params']['p_ex'])
RATE.append(optimizer_21.max['params']['rate'])
PRIOR.append(optimizer_21.max['params']['prior'])
LL.append(optimizer_21.max['target'])

optimizer_22 = BayesianOptimization(
    f=P_active_inference_subject_22,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_22, logs=["./logs_22.log.json"])
P_AL.append(optimizer_22.max['params']['p_al'])
P_AI.append(optimizer_22.max['params']['p_ai'])
P_EX.append(optimizer_22.max['params']['p_ex'])
RATE.append(optimizer_22.max['params']['rate'])
PRIOR.append(optimizer_22.max['params']['prior'])
LL.append(optimizer_22.max['target'])

optimizer_23 = BayesianOptimization(
    f=P_active_inference_subject_23,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_23, logs=["./logs_23.log.json"])
P_AL.append(optimizer_23.max['params']['p_al'])
P_AI.append(optimizer_23.max['params']['p_ai'])
P_EX.append(optimizer_23.max['params']['p_ex'])
RATE.append(optimizer_23.max['params']['rate'])
PRIOR.append(optimizer_23.max['params']['prior'])
LL.append(optimizer_23.max['target'])

optimizer_24 = BayesianOptimization(
    f=P_active_inference_subject_24,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_24, logs=["./logs_24.log.json"])
P_AL.append(optimizer_24.max['params']['p_al'])
P_AI.append(optimizer_24.max['params']['p_ai'])
P_EX.append(optimizer_24.max['params']['p_ex'])
RATE.append(optimizer_24.max['params']['rate'])
PRIOR.append(optimizer_24.max['params']['prior'])
LL.append(optimizer_24.max['target'])

optimizer_25 = BayesianOptimization(
    f=P_active_inference_subject_25,
    pbounds=pbounds,
    random_state=1,allow_duplicate_points=True)
load_logs(optimizer_25, logs=["./logs_25.log.json"])
P_AL.append(optimizer_25.max['params']['p_al'])
P_AI.append(optimizer_25.max['params']['p_ai'])
P_EX.append(optimizer_25.max['params']['p_ex'])
RATE.append(optimizer_25.max['params']['rate'])
PRIOR.append(optimizer_25.max['params']['prior'])
LL.append(optimizer_25.max['target'])


Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 1 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 2 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 3 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 4 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 5 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 6 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 7 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 8 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 9 duplicates registered. Continuing ...
Data point [1.e-03 1.e-03 1.e+01 1.e+01 1.e-03] is not unique. 10 duplica

In [49]:
for i in range(25):
    print("LL SUB("+str(i+1)+"):",LL[i])

LL SUB(1): -107.78329458104761
LL SUB(2): -95.26490722240243
LL SUB(3): -51.98281752333076
LL SUB(4): -74.62594750954806
LL SUB(5): -130.00350693013232
LL SUB(6): -73.10579598772996
LL SUB(7): -69.36977502331914
LL SUB(8): -57.29226442706854
LL SUB(9): -96.73009075890894
LL SUB(10): -117.84033425299525
LL SUB(11): -49.062860872931694
LL SUB(12): -66.09041835687981
LL SUB(13): -110.39658917953966
LL SUB(14): -121.08340232058238
LL SUB(15): -83.37861673990274
LL SUB(16): -75.00265618490714
LL SUB(17): -129.14871265460428
LL SUB(18): -34.49265801034529
LL SUB(19): -16.67466311412396
LL SUB(20): -126.50844855539758
LL SUB(21): -59.34097765013917
LL SUB(22): -59.957642944704844
LL SUB(23): -106.40575567763742
LL SUB(24): -33.9628023004862
LL SUB(25): -133.53930268137194


In [52]:
for i in range(25):
    print("P_AL SUB("+str(i+1)+"):",P_AL[i])
    print("P_AI SUB("+str(i+1)+"):",P_AI[i])
    print("P_EX SUB("+str(i+1)+"):",P_EX[i])
    print("RATE SUB("+str(i+1)+"):",RATE[i])
    print("PRIOR SUB("+str(i+1)+"):",PRIOR[i])
    print('')

P_AL SUB(1): 0.001
P_AI SUB(1): 3.0839226087548943
P_EX SUB(1): 10.0
RATE SUB(1): 8.460884466391647
PRIOR SUB(1): 8.89655518389068

P_AL SUB(2): 0.001
P_AI SUB(2): 1.191055864255718
P_EX SUB(2): 5.412534811547436
RATE SUB(2): 7.04314494537081
PRIOR SUB(2): 0.001

P_AL SUB(3): 4.115987213671824
P_AI SUB(3): 7.5164884902954485
P_EX SUB(3): 5.319828768607106
RATE SUB(3): 4.507510800031042
PRIOR SUB(3): 1.3161116083360356

P_AL SUB(4): 4.223533346254252
P_AI SUB(4): 0.001
P_EX SUB(4): 10.0
RATE SUB(4): 10.0
PRIOR SUB(4): 6.207882389958047

P_AL SUB(5): 4.5722370688021785
P_AI SUB(5): 0.001
P_EX SUB(5): 10.0
RATE SUB(5): 0.001
PRIOR SUB(5): 1.9455722962842914

P_AL SUB(6): 10.0
P_AI SUB(6): 3.1528940931321685
P_EX SUB(6): 10.0
RATE SUB(6): 10.0
PRIOR SUB(6): 7.403184296498887

P_AL SUB(7): 3.1046037683191914
P_AI SUB(7): 3.1751477634538863
P_EX SUB(7): 5.715494870958846
RATE SUB(7): 7.9024197644208085
PRIOR SUB(7): 1.2397617028162318

P_AL SUB(8): 9.976372528762845
P_AI SUB(8): 5.5463788448

In [50]:
def P_active_inference_test(RATE,PRIOR,P_AL,P_AI,P_EX):
    trial_num = 120
    subject_num = 25
    if_can_ask_sub = []
    action_stay_cue_sub = []
    result_stay_cue_sub = []
    action_safe_risk_sub = []
    result_safe_risk_sub = []
    for i in range(subject_num):
        if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(i)
        if_can_ask_sub.append(if_can_ask)
        action_stay_cue_sub.append(action_stay_cue)
        result_stay_cue_sub.append(result_stay_cue)
        action_safe_risk_sub.append(action_safe_risk)
        result_safe_risk_sub.append(result_safe_risk)
    rate = RATE
    prior = PRIOR
    p_al = P_AL
    p_ai = P_AI
    p_ex = P_EX

    LL = []
    for i in range(subject_num):
        log_p_policy = 0
        a = np.array([[100.0,100,prior[i],prior[i],0,0,0,0],
                  [0,0,prior[i],prior[i],0,0,0,0],
                  [0,0,prior[i],prior[i],0,0,0,0],
                  [0,0,prior[i],prior[i],0,0,0,0],
                  [0,0,prior[i],prior[i],0,0,0,0],
                  [0,0,0,0,100,100,0,0],
                  [0,0,0,0,0,0,100,0],
                  [0,0,0,0,0,0,0,100]])
        A = dir(a)
        for j in range(trial_num):
            log_p_policy += np.log(P_stay_cue(A,a,action_stay_cue_sub[i][j],if_can_ask_sub[i][j],p_al[i],p_ai[i],p_ex[i]))
            log_p_policy += np.log(P_safe_risk(A,a,action_safe_risk_sub[i][j],result_stay_cue_sub[i][j],p_al[i],p_ai[i],p_ex[i]))
            a = a_update(a,result_stay_cue_sub[i][j],result_safe_risk_sub[i][j],action_safe_risk_sub[i][j],rate[i])
            A = dir(a)
        LL.append(log_p_policy)
    return LL
LL_1 = P_active_inference_test(RATE,PRIOR,P_AL,P_AI,P_EX)
for i in range(25):
    print("LL SUB("+str(i+1)+"):",LL_1[i])

LL SUB(1): -107.78329458104761
LL SUB(2): -95.26490722240243
LL SUB(3): -51.98281752333076
LL SUB(4): -74.62594750954806
LL SUB(5): -130.00350693013232
LL SUB(6): -73.10579598772996
LL SUB(7): -69.36977502331914
LL SUB(8): -57.29226442706854
LL SUB(9): -96.73009075890894
LL SUB(10): -117.84033425299525
LL SUB(11): -49.062860872931694
LL SUB(12): -66.09041835687981
LL SUB(13): -110.39658917953966
LL SUB(14): -121.08340232058238
LL SUB(15): -83.37861673990274
LL SUB(16): -75.00265618490714
LL SUB(17): -129.14871265460428
LL SUB(18): -34.49265801034529
LL SUB(19): -16.67466311412396
LL SUB(20): -126.50844855539758
LL SUB(21): -59.34097765013917
LL SUB(22): -59.95764294470485
LL SUB(23): -106.40575567763742
LL SUB(24): -33.96280230048621
LL SUB(25): -133.53930268137194
